In [ ]:
# Cell 1: Import libraries and setup

import os
import sys
import yaml
import numpy as np
from nuscenes.nuscenes import NuScenes
import matplotlib.pyplot as plt
from IPython.display import HTML

# Add project root to sys.path
NOTEBOOK_DIR = os.path.abspath('')
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print(f"Added project root to sys.path: {PROJECT_ROOT}")

# Import utility modules
from utils.nuscenes_utils import (
    load_nuscenes_instance, 
    find_dynamic_instances_in_scene,
    get_annotations_with_timestamps
)
from utils.interpolation import interpolate_sequence
from utils.visualization import create_animation_with_lidar
from utils.data_processing import process_instance_sequence

# Load config
try:
    with open(os.path.join(PROJECT_ROOT, 'config/m_detector_config.yaml'), 'r') as f:
        config = yaml.safe_load(f)
except FileNotFoundError:
    print("Config file not found. Using default settings.")
    config = {'nuscenes': {'version': 'v1.0-mini', 'dataroot': '/path/to/nuscenes'}}

# Initialize NuScenes
nusc = NuScenes(
    version=config['nuscenes']['version'],
    dataroot=config['nuscenes']['dataroot'],
    verbose=True
)

print("NuScenes and helper functions loaded.")

In [ ]:
# Cell 2: Select a scene and find dynamic instances

# Choose a scene
scene_index = 1  # Change as needed
my_scene = nusc.scene[scene_index]
print(f"Selected Scene: {my_scene['name']} (Description: {my_scene['description']})")

# Find dynamic instances in the scene
min_annotations = 5
dynamic_instances = find_dynamic_instances_in_scene(nusc, my_scene['token'], min_annotations)

print(f"Found {len(dynamic_instances)} dynamic instances with at least {min_annotations} annotations:")
for idx, instance_token in enumerate(dynamic_instances[:5]):  # Show first 5
    instance = nusc.get('instance', instance_token)
    category = nusc.get('category', instance['category_token'])
    print(f"  {idx+1}. {category['name']} with {instance['nbr_annotations']} annotations")

if dynamic_instances:
    # Select the first instance for demonstration
    selected_instance_token = dynamic_instances[0]
else:
    print("No suitable instances found. Try another scene or lower the annotation threshold.")

In [ ]:
# Cell 3: Process the selected instance

if 'selected_instance_token' in locals():
    # Load instance data
    instance_data = load_nuscenes_instance(nusc, selected_instance_token)
    
    if instance_data:
        print(f"\nProcessing instance: {instance_data['category_name']} with "
              f"{instance_data['nbr_annotations']} annotations")
        
        # Get keyframe boxes and timestamps
        keyframe_boxes, keyframe_timestamps = get_annotations_with_timestamps(nusc, instance_data)
        
        # Interpolate the sequence
        target_frequency = 10  # Hz
        interpolated_boxes, interpolated_timestamps = interpolate_sequence(
            keyframe_boxes, keyframe_timestamps, target_frequency
        )
        
        print(f"Generated {len(interpolated_boxes)} interpolated boxes "
              f"from {len(keyframe_boxes)} keyframes")
        
        # Get LiDAR sweeps for visualization
        from utils.nuscenes_utils import get_lidar_sweeps_for_interval
        
        first_sample_token = instance_data['annotations'][0]['sample_token']
        last_sample_token = instance_data['annotations'][-1]['sample_token']
        
        lidar_sweeps = get_lidar_sweeps_for_interval(nusc, first_sample_token, last_sample_token)
        print(f"Found {len(lidar_sweeps)} LiDAR sweeps for visualization")

In [ ]:
# # Cell 4: Interpolate boxes to exactly match LiDAR timestamps and visualize

# from utils.visualization import create_exact_synchronized_animation

# # Get all LiDAR sweeps for the scene
all_lidar_sweeps = get_lidar_sweeps_for_interval(nusc, my_scene['first_sample_token'], my_scene['last_sample_token'])
print(f"Found {len(all_lidar_sweeps)} LiDAR sweeps in the scene")

# # Create animation with exact synchronization
# animation = create_exact_synchronized_animation(
#     nusc,
#     instance_token,
#     all_lidar_sweeps,
#     interval_ms=100,
#     figsize=(10, 10),
#     point_downsample=20
# )

# # Display animation
# if animation:
#     display(animation)

In [ ]:
# Cell 5: Create multi-box animation with exact synchronization
from utils.visualization import create_multi_box_exact_synchronized_animation

plt.rcParams['animation.embed_limit'] = 2**128 # allow large animation

# Find dynamic instances in the scene
dynamic_instances = find_dynamic_instances_in_scene(nusc, my_scene['token'], min_annotations=2)
print(f"Found {len(dynamic_instances)} dynamic instances")

# Create animation with multiple boxes
multi_box_animation = create_multi_box_exact_synchronized_animation(
    nusc, dynamic_instances, all_lidar_sweeps,
    interval_ms=50, # Corresponds to 20 FPS for display
    save_path="output_animation.mp4",
    save_writer="ffmpeg",
    save_fps=20, # Save at 20 FPS
    save_dpi=200
)




In [ ]:

# Display animation
if multi_box_animation:
    display(multi_box_animation)